In [9]:
# === Imports (external dependencies) ===
import os, sys
sys.path.insert(0, os.path.abspath('..'))          # repo root on the import path

import ee
import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold          # spatial blocks — NOT a random split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, f1_score

import col_amazon_fire_utils as utils

ee = utils.initialize_ee()
nucleus_geom = utils.get_nucleus_geometry()

# MapBiomas LULC — only to derive the 'mosaic' distance
mb_lulc = ee.Image('projects/mapbiomas-colombia/assets/LULC/COLECCION3/INTEGRACION/COLOMBIA-1')
from_codes = [3, 6, 5,   4, 11, 12, 13,   15,   18, 35,   21,   33,   9, 23, 24, 29, 30, 68]
to_groups  = [1, 1, 1,   2,  2,  2,  2,    3,    4,  4,    5,    6,    7,  7,  7,  7,  7,  7]

# Anthropogenic vector assets (time-invariant, present conditions)
base = 'projects/col-amazon-fire-susceptibility/assets/'
layers = {
    'parks': ee.FeatureCollection(base + 'national_natural_parks'),
    'roads': ee.FeatureCollection(base + 'osm_roads_v2'),
    'coca':  ee.FeatureCollection(base + 'coca_cultivation_v2'),
}
print("Setup ready.")

Setup ready.


In [10]:
# === Distance predictors — SAME for every year (a road/river/mosaic doesn't move yearly) ===
SEARCH_RADIUS = 10000
buf = nucleus_geom.buffer(SEARCH_RADIUS)

dist_roads = (layers['roads'].filterBounds(buf)
              .distance(searchRadius=SEARCH_RADIUS).rename('dist_roads'))

dist_parks = (layers['parks'].filterBounds(nucleus_geom.buffer(100000))
              .distance(searchRadius=100000).rename('dist_parks'))   # larger radius: avoidance is broad-scale

def coca_ever(f):                                     # coca = historical presence (any year 2001–2023)
    total = ee.List([ee.Number(f.get(f'coca_{y}')) for y in range(2001, 2024)]).reduce(ee.Reducer.sum())
    return f.set('coca_total', total)
coca_any = layers['coca'].filterBounds(buf).map(coca_ever).filter(ee.Filter.gt('coca_total', 0))
dist_coca = coca_any.distance(searchRadius=SEARCH_RADIUS).rename('dist_coca')

mosaic_mask = (mb_lulc.select('classification_2024').clip(buf)
               .remap(from_codes, to_groups, 0).eq(5))
dist_mosaic = (mosaic_mask.selfMask().fastDistanceTransform().sqrt()
               .multiply(ee.Image.pixelArea().sqrt()).rename('dist_mosaic'))

dist_stack = dist_roads.addBands(dist_parks).addBands(dist_coca).addBands(dist_mosaic).clip(nucleus_geom)
print("Distance predictors:", dist_stack.bandNames().getInfo())

Distance predictors: ['dist_roads', 'dist_parks', 'dist_coca', 'dist_mosaic']


In [11]:
# === Per-year dry-season climate + per-year burned target (the temporal axis) ===
def climate_year(year):
    """Dry-season (Dec[y-1]–Feb[y]) climate as an image for ONE year."""
    start = ee.Date.fromYMD(year - 1, 12, 1)
    end   = ee.Date.fromYMD(year, 3, 1)
    era5  = ee.ImageCollection('ECMWF/ERA5_LAND/MONTHLY_AGGR').filterDate(start, end)

    def add_derived(img):
        u = img.select('u_component_of_wind_10m'); v = img.select('v_component_of_wind_10m')
        wind = u.hypot(v).rename('wind_ms')
        t  = img.select('temperature_2m').subtract(273.15)
        td = img.select('dewpoint_temperature_2m').subtract(273.15)
        es = t.expression('0.6108*exp(17.27*T/(T+237.3))', {'T': t})
        e  = td.expression('0.6108*exp(17.27*Td/(Td+237.3))', {'Td': td})
        rh  = e.divide(es).multiply(100).rename('rh_pct')
        vpd = es.subtract(e).rename('vpd_kPa')
        return img.addBands([wind, rh, vpd])

    era5 = era5.map(add_derived)
    temp   = era5.select('temperature_2m').mean().subtract(273.15).rename('temp_C')
    precip = era5.select('total_precipitation_sum').sum().multiply(1000).rename('precip_mm')
    wind   = era5.select('wind_ms').mean()
    rh     = era5.select('rh_pct').mean()
    vpd    = era5.select('vpd_kPa').mean()
    ndvi   = (ee.ImageCollection('MODIS/061/MOD13A1').filterDate(start, end)
              .select('NDVI').mean().multiply(0.0001).rename('ndvi'))
    return temp.addBands([precip, wind, rh, vpd, ndvi])

def burned_year(year):
    """1 if the pixel burned in that dry season, else 0. unmask(0) is ESSENTIAL:
       it turns 'never burned' pixels into explicit zeros so sampling keeps them."""
    return (ee.ImageCollection('MODIS/061/MCD64A1')
            .filterDate(ee.Date.fromYMD(year - 1, 12, 1), ee.Date.fromYMD(year, 4, 1))
            .select('BurnDate').max().gt(0).unmask(0).rename('burned'))

In [12]:
# === Fixed sample sites: distances + coordinates, sampled ONCE ===
# These locations stay the same across all years; only climate/burned change per year.
sites_fc = dist_stack.sample(
    region=nucleus_geom, scale=500, numPixels=3000, seed=42, geometries=True)
print("Sample sites:", sites_fc.size().getInfo())

Sample sites: 1293


In [13]:
# === Build the pixel-year table: loop over years, sample climate+burned at the fixed sites ===
DATA = '../datasets/'
pixel_year_csv = DATA + 'pixel_year_df.csv'

def fc_to_df_xy(fc):
    feats = fc.getInfo()['features']
    rows = []
    for f in feats:
        row = dict(f['properties'])
        if f.get('geometry'):
            row['lon'], row['lat'] = f['geometry']['coordinates']
        rows.append(row)
    return pd.DataFrame(rows)

if os.path.exists(pixel_year_csv):
    pixel_year_df = pd.read_csv(pixel_year_csv)
    print("Loaded pixel_year_df from CSV.")
else:
    frames = []
    for y in range(2001, 2025):                          # dry season of each year
        img_y = climate_year(y).addBands(burned_year(y))
        # sampleRegions samples img_y AT the fixed sites and CARRIES OVER the distance
        # properties, so each row is already complete: distances + that year's climate + burned
        sampled = img_y.sampleRegions(collection=sites_fc, scale=500, geometries=True)
        df_y = fc_to_df_xy(sampled)
        df_y['year'] = y
        frames.append(df_y)
        print(f"  {y}: {len(df_y)} rows, burned={int(df_y['burned'].sum())}")

    pixel_year_df = pd.concat(frames, ignore_index=True)
    os.makedirs(DATA, exist_ok=True)
    pixel_year_df.to_csv(pixel_year_csv, index=False)
    print("Saved pixel_year_df.")

print("\nTotal rows:", len(pixel_year_df),
      "| burned:", int(pixel_year_df['burned'].sum()),
      f"({100*pixel_year_df['burned'].mean():.1f}%)")

  2001: 1293 rows, burned=21
  2002: 1293 rows, burned=13
  2003: 1293 rows, burned=14
  2004: 1293 rows, burned=94
  2005: 1293 rows, burned=10
  2006: 1293 rows, burned=24
  2007: 1293 rows, burned=115
  2008: 1293 rows, burned=36
  2009: 1293 rows, burned=30
  2010: 1293 rows, burned=22
  2011: 1293 rows, burned=49
  2012: 1293 rows, burned=14
  2013: 1293 rows, burned=21
  2014: 1293 rows, burned=46
  2015: 1293 rows, burned=9
  2016: 1293 rows, burned=27
  2017: 1293 rows, burned=22
  2018: 1293 rows, burned=114
  2019: 1293 rows, burned=9
  2020: 1293 rows, burned=19
  2021: 1293 rows, burned=9
  2022: 1293 rows, burned=67
  2023: 1293 rows, burned=5
  2024: 1293 rows, burned=5
Saved pixel_year_df.

Total rows: 31032 | burned: 795 (2.6%)


In [14]:
# === Spatial blocks + spatial block cross-validation (baseline) ===
df = pixel_year_df.dropna().copy()

# Assign each site to a spatial block from its coordinates (fixed across years).
# GroupKFold keeps every row of a block (all its years) in the SAME fold -> tests
# generalisation to NEW places, not to neighbours. This removes the inflation of a random split.
BLOCK = 0.25   # ~27 km blocks; should exceed the fire autocorrelation range (confirm later with Moran's I)
df['block'] = (df['lon'] // BLOCK).astype(int).astype(str) + '_' + \
              (df['lat'] // BLOCK).astype(int).astype(str)
print("Spatial blocks:", df['block'].nunique())

pred_cols = ['dist_roads','dist_parks','dist_coca','dist_mosaic',
             'temp_C','precip_mm','wind_ms','rh_pct','vpd_kPa','ndvi']
X = df[pred_cols].values
y = df['burned'].astype(int).values
groups = df['block'].values

gkf = GroupKFold(n_splits=5)
aucs, f1s = [], []
for fold, (tr, te) in enumerate(gkf.split(X, y, groups), 1):
    sc = StandardScaler().fit(X[tr])                     # scaler fit ONLY on train — no leakage
    m = LogisticRegression(max_iter=1000, class_weight='balanced').fit(sc.transform(X[tr]), y[tr])
    p = m.predict_proba(sc.transform(X[te]))[:, 1]
    aucs.append(roc_auc_score(y[te], p))
    f1s.append(f1_score(y[te], m.predict(sc.transform(X[te]))))
    print(f"  Fold {fold}: AUC={aucs[-1]:.3f}  F1={f1s[-1]:.3f}")

print(f"\nSPATIAL BLOCK CV — baseline logistic")
print(f"  AUC-ROC : {np.mean(aucs):.3f} ± {np.std(aucs):.3f}")
print(f"  F1-score: {np.mean(f1s):.3f} ± {np.std(f1s):.3f}")

Spatial blocks: 110
  Fold 1: AUC=0.817  F1=0.133
  Fold 2: AUC=0.816  F1=0.138
  Fold 3: AUC=0.884  F1=0.180
  Fold 4: AUC=0.827  F1=0.155
  Fold 5: AUC=0.754  F1=0.076

SPATIAL BLOCK CV — baseline logistic
  AUC-ROC : 0.819 ± 0.041
  F1-score: 0.136 ± 0.034


In [15]:
# === Temporal validation: train on old years, test on recent years ===
# Answers "does it predict years it never saw?" — the honesty check for stakeholders.
train = df['year'] <= 2019
test  = df['year'] >= 2020

Xtr, ytr = df.loc[train, pred_cols].values, df.loc[train, 'burned'].astype(int).values
Xte, yte = df.loc[test,  pred_cols].values, df.loc[test,  'burned'].astype(int).values

sc = StandardScaler().fit(Xtr)
m = LogisticRegression(max_iter=1000, class_weight='balanced').fit(sc.transform(Xtr), ytr)
p = m.predict_proba(sc.transform(Xte))[:, 1]

print("TEMPORAL VALIDATION — baseline logistic")
print(f"  train ≤2019: {train.sum()} rows | test ≥2020: {test.sum()} rows")
print(f"  AUC-ROC : {roc_auc_score(yte, p):.3f}")
print(f"  F1-score: {f1_score(yte, m.predict(sc.transform(Xte))):.3f}")

TEMPORAL VALIDATION — baseline logistic
  train ≤2019: 24567 rows | test ≥2020: 6465 rows
  AUC-ROC : 0.824
  F1-score: 0.074
